# Chapter 03: Building an Inverted Index

Postings, OR union, manifest references and the eligible view.

In [ ]:
import sys
from pathlib import Path

# Resolve the repository root by walking parents from the current working directory.
ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "content" / "books" / "retrieval" / "_index.md").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError("repository root not found")

# Make this notebook's shared helper importable, then re-resolve through it.
NOTEBOOK_DIR = ROOT / "notebooks" / "books" / "retrieval"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import _support
ROOT = _support.repo_root()
print("repo root:", ROOT)

## Build an index from the eligible baseline view

In [ ]:
from document_identity import Collection
from lexical_index import InvertedIndex

collection = Collection.load(ROOT / "examples" / "fixtures" / "part1-v1.json")
index = InvertedIndex.from_collection(collection, "baseline")
manifest = index.manifest()
print("rows:", len(manifest["rows"]), "terms:", len(manifest["postings"]), "fingerprint:", manifest["fingerprint"])

## Inspect one posting list

In [ ]:
info = index.inspect("cache")
print("term:", info["term"], "df:", info["df"])
for posting in info["postings"]:
    print("row", posting["row"], "tf", posting["tf"], "positions", posting["positions"])
    print("  passage:", posting["reference"]["passage_id"])

assert info["term"] == "cache"
assert info["df"] == 3
assert sum(p["tf"] for p in info["postings"]) == 11

## OR union at query time

In [ ]:
result = index.search("cache reset", method="overlap", limit=10)
print("visited postings entries:", result["work"]["postings_entries_visited"])
print("candidates:", [(r["passage_id"], r["score"]) for r in result["retrieved"]])
assert result["work"]["postings_entries_visited"] == 8
assert result["status"] == "candidates"

## Reconstruct a reference from a row

In [ ]:
row_id = result["retrieved"][0]["row"]
reference = index.reference(row_id)
print(reference)
resolved = collection.resolve_history(reference, "baseline")
print("resolved:", resolved["status"])
assert resolved["status"] == "resolved"
assert resolved["record"]["passage_id"] == result["retrieved"][0]["passage_id"]

## Eligibility is baked into the indexed view

In [ ]:
# A restricted snapshot yields no eligible passages and therefore no postings.
restricted_index = InvertedIndex.from_collection(collection, "removed")
print("removed-view rows:", len(restricted_index.manifest()["rows"]))
assert len(restricted_index.manifest()["rows"]) == 0

## Try it yourself
Inspect `reset`, change the query, or rebuild from `both-eligible`. Compare visited posting entries with passages scanned by the scan method.

In [ ]:
scan_result = index.search("cache reset", method="scan", limit=10)
print("scan passages_scanned:", scan_result["work"]["passages_scanned"])
print("overlap visited:", result["work"]["postings_entries_visited"])

## Observed versus not established
Observed: postings, OR union, manifest reconstruction and view filtering. Not established: latency, memory, throughput or general performance gain.